# 3.01 — Baseline: DummyClassifier
**Etapa 1 · Tech Challenge POSTECH MLOps**

Estabelece o **piso absoluto** de performance usando a estratégia `most_frequent`.

### Rastreamento MLflow
Este notebook utiliza o backend **filesystem** (`mlruns/`), seguindo o padrão
do projeto referência do professor.  
A pasta `mlruns/` deve ser **incluída no controle de versão** (exceção no `.gitignore`)
para que os baselines possam ser compartilhados entre membros do grupo:

```gitignore
# .gitignore — adicionar exceção para compartilhar baselines
mlruns/          # ← remover esta linha, OU adicionar abaixo:
# !mlruns/       # ← descomentar para commitar mlruns/
mlartifacts/     # ← artefatos binários grandes — manter ignorado
```

> **Estrutura gerada em `mlruns/`**
> ```
> mlruns/
> └── <experiment_id>/
>     ├── meta.yaml
>     └── <run_id>/
>         ├── meta.yaml
>         ├── metrics/          ← um arquivo por métrica (commitável)
>         │   ├── cv_roc_auc_mean
>         │   ├── cv_average_precision_mean
>         │   └── ...
>         ├── params/           ← um arquivo por parâmetro
>         └── artifacts/
>             └── figures/
>                 ├── dummy_confusion_matrix.png
>                 └── dummy_roc_curve.png
> ```


## 0. Setup

In [1]:
"""Configuração de logging estruturado, seeds e supressão de warnings esperados."""
from __future__ import annotations

import logging
import sys
import warnings
from pathlib import Path

# ── Root do projeto (notebooks/ está um nível abaixo do root) ────────────────
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# ── Logging estruturado — NENHUM print() no projeto ──────────────────────────
(ROOT / "logs").mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(ROOT / "logs" / "3.01_baseline_dummy.log", mode="w"),
    ],
)
logger = logging.getLogger("baseline.dummy")

# Suprime FutureWarning do filesystem backend do MLflow
# (comportamento esperado; usamos filesystem para compatibilidade com o padrão do projeto)
warnings.filterwarnings(
    "ignore",
    message="The filesystem tracking backend.*is deprecated",
    category=FutureWarning,
)
warnings.filterwarnings("ignore", category=UserWarning)

logger.info("ROOT            : %s", ROOT)
logger.info("Python          : %s", sys.version.split()[0])


2026-04-14 17:17:06 | INFO     | baseline.dummy | ROOT            : c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\projeto_final
2026-04-14 17:17:06 | INFO     | baseline.dummy | Python          : 3.10.20


In [3]:
"""Importações e seed global."""
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

from churn_telecom.config import RANDOM_STATE, TARGET

import matplotlib
matplotlib.use("Agg")          # headless — sem display necessário
import matplotlib.pyplot as plt

np.random.seed(RANDOM_STATE)
logger.info("RANDOM_STATE    : %d  (seed fixado)", RANDOM_STATE)
logger.info("MLflow version  : %s", mlflow.__version__)


2026-04-14 17:18:06 | INFO     | baseline.dummy | RANDOM_STATE    : 42  (seed fixado)
2026-04-14 17:18:06 | INFO     | baseline.dummy | MLflow version  : 3.11.1


## 1. Caminhos

| Variável | Caminho | Descrição |
|---|---|---|
| `TRAIN_PATH` | `data/processed/train.parquet` | Split treino (80%, estratificado) |
| `TEST_PATH` | `data/processed/test.parquet` | Split teste (20%, estratificado) |
| `PREPROCESSOR_PATH` | `models/preprocessor.pkl` | ColumnTransformer fitado no treino |
| `MLFLOW_URI` | `mlruns/` | Filesystem backend — commitável |
| `FIGURES_DIR` | `reports/figures/baselines/` | PNGs gerados como artefatos |


In [4]:
TRAIN_PATH        = ROOT / "data" / "processed" / "train.parquet"
TEST_PATH         = ROOT / "data" / "processed" / "test.parquet"
PREPROCESSOR_PATH = ROOT / "models" / "preprocessor.pkl"

# ── MLflow: filesystem backend (padrão do projeto do professor) ───────────────
# Gera mlruns/<exp_id>/<run_id>/{metrics,params,artifacts}/
# Para compartilhar via git: remova 'mlruns/' do .gitignore (ou adicione '!mlruns/')
MLFLOW_URI      = str(ROOT / "mlruns")
EXPERIMENT_NAME = "churn-telecom"

FIGURES_DIR = ROOT / "reports" / "figures" / "baselines"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Validação de pré-requisitos ───────────────────────────────────────────────
_missing = [p for p in (TRAIN_PATH, TEST_PATH, PREPROCESSOR_PATH) if not p.exists()]
if _missing:
    for p in _missing:
        logger.error("FALTANDO: %s", p)
    raise FileNotFoundError(
        f"Execute os notebooks 1.01→1.03 antes deste. Faltando: {_missing}"
    )

for p in (TRAIN_PATH, TEST_PATH, PREPROCESSOR_PATH):
    logger.info("OK  %s", p.relative_to(ROOT))


2026-04-14 17:18:28 | INFO     | baseline.dummy | OK  data\processed\train.parquet
2026-04-14 17:18:28 | INFO     | baseline.dummy | OK  data\processed\test.parquet
2026-04-14 17:18:28 | INFO     | baseline.dummy | OK  models\preprocessor.pkl


## 2. Carregamento dos Dados

In [5]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_test,  y_test  = test.drop(columns=[TARGET]),  test[TARGET]

preprocessor = joblib.load(PREPROCESSOR_PATH)

logger.info(
    "Train : %d amostras | Churn=%.2f%%  (pos=%d  neg=%d)",
    len(X_train), y_train.mean() * 100,
    int(y_train.sum()), int((y_train == 0).sum()),
)
logger.info(
    "Test  : %d amostras | Churn=%.2f%%",
    len(X_test), y_test.mean() * 100,
)
logger.info("Preprocessor carregado: %s", PREPROCESSOR_PATH.relative_to(ROOT))


KeyError: "['Churn Value'] not found in axis"

## 3. Pipeline — DummyClassifier

In [ ]:
"""
Pipeline: preprocessor (já fitado no treino) → DummyClassifier.

DummyClassifier(strategy='most_frequent'):
  - Prediz SEMPRE a classe majoritária (Churn=0).
  - Representa o comportamento de uma empresa que ignora todos os clientes.
  - É o piso absoluto: qualquer modelo real deve superá-lo amplamente.
"""
dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",   DummyClassifier(strategy="most_frequent",
                                     random_state=RANDOM_STATE)),
])

logger.info("Pipeline construído: %s", [s[0] for s in dummy_pipeline.steps])


## 4. Validação Cruzada Estratificada (5-fold)

> ⚠️ **Nomenclatura sklearn → MLflow**
>
> O sklearn chama PR-AUC de **`average_precision`** (internamente usa `average_precision_score`).
> O `cross_validate` retorna a chave `"test_average_precision"`.
> Após strip do prefixo, o MLflow recebe **`cv_average_precision_mean`** — nunca `cv_pr_auc_mean`.
>
> | Conceito | `scoring=` | Chave em `mlruns/metrics/` |
> |---|---|---|
> | AUC-ROC | `"roc_auc"` | `cv_roc_auc_mean` |
> | PR-AUC  | `"average_precision"` | `cv_average_precision_mean` |
> | F1      | `"f1"` | `cv_f1_mean` |
> | Recall  | `"recall"` | `cv_recall_mean` |
> | Precision | `"precision"` | `cv_precision_mean` |
> | MCC     | `"matthews_corrcoef"` | `cv_matthews_corrcoef_mean` |


In [ ]:
SCORING = [
    "roc_auc",
    "average_precision",   # PR-AUC — nome canônico do sklearn (NÃO "pr_auc")
    "f1",
    "recall",
    "precision",
    "matthews_corrcoef",
]

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logger.info("Iniciando cross_validate | strategy=most_frequent | folds=5 | stratified=True")

cv_results = cross_validate(
    dummy_pipeline,
    X_train, y_train,
    cv=CV,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

logger.info("cross_validate concluído.")
for key, values in cv_results.items():
    if key.startswith("test_"):
        name = key.removeprefix("test_")
        logger.info(
            "  cv %-26s  mean=%+.4f  std=%.4f",
            name, values.mean(), values.std(),
        )


## 5. Registro no MLflow (filesystem backend)

Os dados são gravados em `mlruns/` como arquivos individuais:
- `mlruns/<exp_id>/<run_id>/metrics/cv_roc_auc_mean`
- `mlruns/<exp_id>/<run_id>/params/model_type`
- `mlruns/<exp_id>/<run_id>/artifacts/figures/dummy_confusion_matrix.png`

Para **compartilhar os baselines via git**, adicione ao `.gitignore` do projeto:
```gitignore
# Permitir rastreamento dos baselines
!mlruns/
```


In [ ]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

logger.info("MLflow tracking URI : %s", MLFLOW_URI)
logger.info("Experiment          : %s", EXPERIMENT_NAME)

with mlflow.start_run(run_name="dummy_most_frequent") as run:
    RUN_ID = run.info.run_id
    EXP_ID = run.info.experiment_id
    logger.info("Run iniciado | run_id=%s | exp_id=%s", RUN_ID, EXP_ID)

    # ── 5.1 Parâmetros ────────────────────────────────────────────────────
    mlflow.log_params({
        "model_type":       "DummyClassifier",
        "strategy":         "most_frequent",
        "random_state":     RANDOM_STATE,
        "cv_folds":         5,
        "cv_stratified":    True,
        "dataset_train":    str(TRAIN_PATH.relative_to(ROOT)),
        "dataset_test":     str(TEST_PATH.relative_to(ROOT)),
        "n_train":          len(X_train),
        "n_test":           len(X_test),
        "churn_rate_train": round(float(y_train.mean()), 4),
    })

    # ── 5.2 Métricas CV ───────────────────────────────────────────────────
    # Chaves geradas: cv_roc_auc_mean, cv_average_precision_mean, cv_f1_mean ...
    # Cada uma vira um arquivo em mlruns/<exp_id>/<run_id>/metrics/
    for key, values in cv_results.items():
        if key.startswith("test_"):
            name = key.removeprefix("test_")
            mlflow.log_metric(f"cv_{name}_mean", float(values.mean()))
            mlflow.log_metric(f"cv_{name}_std",  float(values.std()))

    # ── 5.3 Métricas Hold-out ─────────────────────────────────────────────
    dummy_pipeline.fit(X_train, y_train)
    y_pred  = dummy_pipeline.predict(X_test)
    y_proba = dummy_pipeline.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    mlflow.log_metric("test_roc_auc",
        float(roc_auc_score(y_test, y_proba)))
    mlflow.log_metric("test_average_precision",
        float(average_precision_score(y_test, y_proba)))
    mlflow.log_metric("test_f1",
        float(f1_score(y_test, y_pred, zero_division=0)))
    mlflow.log_metric("test_recall",
        float(recall_score(y_test, y_pred, zero_division=0)))
    mlflow.log_metric("test_precision",
        float(precision_score(y_test, y_pred, zero_division=0)))
    mlflow.log_metric("test_mcc",
        float(matthews_corrcoef(y_test, y_pred)))
    mlflow.log_metric("test_npv",
        float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0)

    logger.info(
        "Hold-out | roc_auc=%.4f | avg_precision=%.4f | f1=%.4f | recall=%.4f",
        roc_auc_score(y_test, y_proba),
        average_precision_score(y_test, y_proba),
        f1_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
    )

    # ── 5.4 Artefatos (figuras) ───────────────────────────────────────────
    # Gravados em mlruns/<exp_id>/<run_id>/artifacts/figures/
    # mlartifacts/ é usado apenas com mlflow server --artifacts-destination

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Não Churn", "Churn"],
        colorbar=False, ax=ax,
    )
    ax.set_title("Confusion Matrix — DummyClassifier (most_frequent)", fontsize=10)
    cm_path = FIGURES_DIR / "dummy_confusion_matrix.png"
    fig.savefig(cm_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    mlflow.log_artifact(str(cm_path), artifact_path="figures")
    logger.info("Artefato: %s", cm_path.name)

    # ROC Curve
    fig, ax = plt.subplots(figsize=(5, 5))
    RocCurveDisplay.from_predictions(
        y_test, y_proba, ax=ax, name="DummyClassifier"
    )
    ax.plot([0, 1], [0, 1], "k--", label="Chance (AUC=0.50)")
    ax.set_title("ROC Curve — DummyClassifier", fontsize=10)
    ax.legend(loc="lower right")
    roc_path = FIGURES_DIR / "dummy_roc_curve.png"
    fig.savefig(roc_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    mlflow.log_artifact(str(roc_path), artifact_path="figures")
    logger.info("Artefato: %s", roc_path.name)

    # Modelo serializado
    mlflow.sklearn.log_model(dummy_pipeline, "dummy_classifier")
    logger.info("Modelo logado em mlruns.")

    logger.info(
        "Run finalizado | run_id=%s\n"
        "  Estrutura: mlruns/%s/%s/",
        RUN_ID, EXP_ID, RUN_ID,
    )


## 6. Sumário

In [ ]:
"""Tabela de resultados e validação das chaves no MLflow."""
_roc  = float(roc_auc_score(y_test, y_proba))
_ap   = float(average_precision_score(y_test, y_proba))
_f1   = float(f1_score(y_test, y_pred, zero_division=0))
_rec  = float(recall_score(y_test, y_pred, zero_division=0))
_prec = float(precision_score(y_test, y_pred, zero_division=0))
_mcc  = float(matthews_corrcoef(y_test, y_pred))
_npv  = float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0

summary = pd.DataFrame([{
    "Chave MLflow":        "test_*",
    "AUC-ROC":             _roc,
    "PR-AUC (avg_prec)":   _ap,
    "F1":                  _f1,
    "Recall":              _rec,
    "Precision":           _prec,
    "MCC":                 _mcc,
    "NPV":                 _npv,
    "Acurácia":            float((tp + tn) / len(y_test)),
}]).set_index("Chave MLflow").round(4)

logger.info("Sumário:\n%s", summary.T.to_string())
summary.T


### Interpretação

| Chave MLflow | Valor | Conclusão |
|---|---|---|
| `test_roc_auc` | 0.500 | Equivalente ao acaso — piso absoluto |
| `test_average_precision` | ≈ 0.265 | ≈ proporção da classe positiva |
| `test_f1` | 0.000 | Nenhum churner detectado |
| `test_recall` | 0.000 | 100% dos churners ignorados → máximo FN |
| Acurácia | ≈ 0.735 | ⚠️ Enganosa — reflete só a proporção da classe dominante |

**Próximo**: `3.02_vab_baseline_logistic.ipynb` — Regressão Logística como baseline linear.
